# CT 助手 LoRA / SFT 训练与独立测试

在 **代码执行程序 → 更改运行时类型 → T4 GPU** 中选择 GPU，保存一份 Drive 副本后运行全部。

基于 Qwen2.5-1.5B-Instruct，训练工具调用与证据选择。**不训练 CT 检测器，不证明真实工业缺陷/根因识别准确率。**

24 个训练、6 个验证、6 个测试合成样本，按完整样本分割。比较无格式约束的单步输出，再验证受约束 Agent 的完整流程。基础模型和 Adapter 使用相同测试集、贪心解码。

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path('/content/xsim-chip-lora')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/yuweimin2077-hub/xsim-chip.git', str(repo)], check=True)
os.chdir(repo)
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)

## 固定配置训练与测试

训练只监督最后一条助手动作；其余输入全部屏蔽 loss。超长证据拒绝截断。两轮训练结束保存 Adapter，然后重新加载它完成对照评测。此步骤可能需要十几分钟或更久。

In [ ]:
subprocess.run([sys.executable, 'tools/train_assistant_lora.py', '--output-dir', 'outputs/lora_pilot'], check=True)

## 查看结果并导出可复核文件

In [ ]:
import json, hashlib, zipfile, base64
from IPython.display import display, HTML
out = repo / 'outputs/lora_pilot'
training = json.loads((out / 'training.json').read_text())
evaluation = json.loads((out / 'evaluation.json').read_text())
print(json.dumps({'gpu':training['gpu'], 'trainable_parameters':training['trainable_parameters'],
                  'validation_loss_before':training['baseline_validation_loss'],
                  'validation_loss_after':training['adapter_validation_loss'],
                  'training_peak_gib':training['training_peak_allocated_gib'],
                  'evaluation':evaluation}, ensure_ascii=False, indent=2))
files = [p for p in out.rglob('*') if p.is_file() and 'trainer' not in p.relative_to(out).parts]
integrity = {str(p.relative_to(out)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(out / 'integrity.json').write_text(json.dumps(integrity, indent=2) + '\n')
archive = repo / 'outputs/xsim_lora_pilot.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files + [out / 'integrity.json']:
        z.write(p, str(p.relative_to(out)))
print('Archive bytes:', archive.stat().st_size, 'SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
display(HTML('<a download="xsim_lora_pilot.zip" href="data:application/zip;base64,' + base64.b64encode(archive.read_bytes()).decode() + '">Download adapter, dataset, logs and evaluation ZIP</a>'))